# Lab 11 — Dimensionality Reduction using Principal Component Analysis (PCA)

### Goal
Implement the key PCA logic yourself and verify the result with `scikit-learn`.

**Student focus:** only the small computational steps marked `TODO`. Plotting and display code is already provided.

### PCA workflow
**Center → Covariance → Eigenvectors/Eigenvalues → Sort → Project → Choose $k$ → Reconstruct → Validate inside CV**

### Important
Do not change the plotting code. Complete only the requested PCA/model logic.


## Step 0. Upload the Dataset
We are running in Google Colab, which does not have access to your local files by default. This cell opens a file-picker dialog so you can upload `student_clean_dataset.csv` into the Colab session.


In [ ]:
import gdown

import os

file_id = "1t5mmVocO1_fGXGqRftpekRom9yxq-ML4"

if not os.path.exists("student_clean_dataset.csv"):

    gdown.download(

        f"https://drive.google.com/uc?id={file_id}",

        "student_clean_dataset.csv",

        quiet=False

    )


## Step 1. Import Libraries

The required libraries are already specified below. Run the cell.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error

# Fix the random seed so splits/results are reproducible on every re-run
np.random.seed(42)


## Step 2. Load the Dataset and Select Two Correlated Features

Use `Study_Hours` and `Exam_Score` as the two features for the manual PCA walkthrough.


In [ ]:
df = pd.read_csv("student_clean_dataset.csv")
print(df.shape)
df.head()


In [ ]:
# Select two correlated numeric features as a 2D array (this plays the role of X in the lab outline)
X2 = df[["Study_Hours", "Exam_Score"]].values.astype(float)

print("Shape of X2:", X2.shape)
print("Correlation between Study_Hours and Exam_Score:",
      np.corrcoef(X2[:, 0], X2[:, 1])[0, 1].round(3))


## Step 3. Visualize the Data Cloud

Run the plotting cell and observe the diagonal shape of the data cloud.


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(X2[:, 0], X2[:, 1], s=20, alpha=0.6, color="steelblue")
plt.xlabel("Study_Hours")
plt.ylabel("Exam_Score")
plt.title("Data Cloud: Study_Hours vs Exam_Score")
plt.axis("equal")
plt.grid(alpha=0.3)
plt.show()


## Step 4. Mean Center the Data

The worksheet formula is:

$$
X_c = X-\mathbf{1}\mu^\top
$$

Here, $\mu$ contains the mean of each feature.

**TODO:** Compute the feature means and create the centered matrix `Xc`.

The centered data should have mean approximately zero in each column.


In [ ]:
# TODO: Compute the feature mean vector mu, center the data, and scale each feature.

#TODO

print("Mean vector (Study_Hours, Exam_Score):", mu)
print("Feature standard deviations:", s)

plt.figure(figsize=(6, 6))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], s=20, alpha=0.6, color="darkorange")
plt.axhline(0, color="gray", linewidth=1)
plt.axvline(0, color="gray", linewidth=1)
plt.xlabel("Study_Hours (standardized)")
plt.ylabel("Exam_Score (standardized)")
plt.title("Centered and Standardized Data Cloud")
plt.axis("equal")
plt.grid(alpha=0.3)
plt.show()


## Step 5. Compute the Covariance Matrix

The worksheet formula is:

$$
S=\frac{1}{n-1}X_c^\top X_c
$$

**TODO:** Implement this matrix formula directly using NumPy.


In [ ]:
# TODO: Compute S using the standardized data:
# TODO

print("Covariance matrix:\n", S)


**Reading the matrix:** `S[0,0]` is the variance of `Study_Hours`, `S[1,1]` is the variance of `Exam_Score`, and `S[0,1] = S[1,0]` is their covariance. A large positive off-diagonal value relative to the diagonal values indicates strong positive correlation — visually, a cloud stretched diagonally rather than a circle.


## Step 6. Find Eigenvalues and Eigenvectors

The worksheet gives:

$$
Sv_j=\lambda_jv_j
$$

and the characteristic equation:

$$
|S-\lambda I|=0
$$

**TODO:** Use `np.linalg.eig(S)` to obtain the eigenvalues and eigenvectors.


In [ ]:
# TODO: Compute the eigenvalues and eigenvectors of S.

print("Eigenvalues:", eigenvalues)
print("Eigenvectors (as columns):\n", eigenvectors)


## Step 7. Sort and Identify the Principal Directions

PCA orders components by decreasing eigenvalue:

$$
\lambda_1\geq\lambda_2\geq\cdots\geq\lambda_p
$$

**TODO:** Sort the eigenpairs from largest to smallest eigenvalue and assign `v1` and `v2`.


In [ ]:
# TODO: Sort the eigenpairs by decreasing eigenvalue and assign v1 and v2.

print("Sorted eigenvalues:", eigenvalues)
print("v1 (1st principal direction):", v1)
print("v2 (2nd principal direction):", v2)


## Step 8. Project the Data onto the Principal Components

The worksheet projection formula is:

$$
z_i=u^\top X_i
$$

For the centered data, use the principal directions `v1` and `v2`.

**TODO:** Compute the two principal-component score vectors.


In [ ]:
# TODO: Project the standardized data onto v1 and v2.

print("First 5 principal-component scores (z1, z2):")
for i in range(5):
    print(f"  Student {i}: z1={z1[i]:.2f}, z2={z2[i]:.2f}")


## Step 9. Visualize the Principal Axes

Run the plotting cell.

Observe how the principal directions are rotated relative to the original axes.


In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], s=20, alpha=0.5, color="steelblue")
plt.quiver(0, 0, *v1, color='r', scale=3, label="v1 (1st principal direction)")
plt.quiver(0, 0, *v2, color='g', scale=3, label="v2 (2nd principal direction)")
plt.xlabel("Study_Hours (standardized)")
plt.ylabel("Exam_Score (standardized)")
plt.title("Principal Axes Overlaid on the Standardized Data Cloud")
plt.axis("equal")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


## Step 10. Verify PCA using scikit-learn

Use `PCA(n_components=2)` and compare its principal directions and explained variance with your manual result.

**TODO:** Create the PCA object and transform `X2`.


In [ ]:
# TODO: Create PCA(n_components=2) and transform the standardized data X_scaled into Z.

print("sklearn principal directions (components_):\n", pca.components_)
print("sklearn explained variance (eigenvalues):", pca.explained_variance_)

print("\nOur manual eigenvalues for comparison:", eigenvalues)


**Sanity check:** `pca.explained_variance_` should closely match our manually computed, sorted `eigenvalues` from Step 7 (values may differ very slightly due to numerical precision, and a component's sign may be flipped — both are expected and harmless).


## Step 11. Explained Variance Ratio

The worksheet defines:

$$
\mathrm{EVR\ of\ PC}_j=
\frac{\lambda_j}{\sum_{\ell=1}^{p}\lambda_\ell}
$$

and cumulative explained variance:

$$
\mathrm{Cumulative\ EVR\ through\ }k=
\frac{\sum_{j=1}^{k}\lambda_j}
{\sum_{\ell=1}^{p}\lambda_\ell}
$$

**TODO:** Obtain the explained-variance ratios and cumulative values from the fitted PCA object.

Then run the supplied plot.


In [ ]:
# TODO: Get explained_variance_ratio_ and compute its cumulative sum.
# The plotting code below is already provided.

print("Explained variance ratio per component:", ratios)
print("Cumulative explained variance:", cumulative)

plt.figure(figsize=(7, 5))
plt.bar(range(1, len(ratios) + 1), ratios, alpha=0.7, label="Individual explained variance")
plt.plot(range(1, len(ratios) + 1), cumulative, marker='o', color='red', label="Cumulative explained variance")
plt.xlabel("Principal Component")
plt.ylabel("Proportion of Variance Explained")
plt.title("Scree Plot: Study_Hours & Exam_Score")
plt.xticks(range(1, len(ratios) + 1))
plt.legend()
plt.grid(alpha=0.3)
plt.show()


With only 2 original features, there are only 2 possible components, so this scree plot is mostly illustrative. The real decision-making happens in the next steps, where we use **all 5 numeric features**, giving PCA more components to choose from.


## Step 12. Move to All Five Numeric Features

Now use:

`Age`, `Study_Hours`, `Attendance`, `Assignments`, `Exam_Score`

The worksheet notes that standardization is useful when features have different scales.

**TODO:**
1. Create `X_full`.
2. Standardize the features with `StandardScaler`.
3. Fit PCA while retaining all possible components.
4. Store the individual and cumulative explained-variance ratios.

Then run the supplied plot.


In [ ]:
feature_cols = ["Age", "Study_Hours", "Attendance", "Assignments", "Exam_Score"]

# TODO: Create X_full, standardize it, fit PCA with all components,
# and compute ratios_full and cumulative_full.

print("Explained variance ratio per component:", np.round(ratios_full, 4))
print("Cumulative explained variance:", np.round(cumulative_full, 4))

plt.figure(figsize=(7, 5))
plt.bar(range(1, len(ratios_full) + 1), ratios_full, alpha=0.7, label="Individual explained variance")
plt.plot(range(1, len(ratios_full) + 1), cumulative_full, marker='o', color='red', label="Cumulative explained variance")
plt.xlabel("Principal Component")
plt.ylabel("Proportion of Variance Explained")
plt.title("Scree Plot: All 5 Numeric Features (Standardized)")
plt.xticks(range(1, len(ratios_full) + 1))
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
threshold = 0.95

# TODO: Find the smallest k for which cumulative_full >= threshold.

print(f"Smallest k reaching {threshold*100:.0f}% cumulative explained variance: k = {k}")


## Step 13. Choose the Number of Components $k$

The worksheet rule is:

> Choose the smallest $k$ reaching a cumulative explained-variance threshold, then validate.

Use a threshold of $0.95$.

**TODO:** Find the first component count for which the cumulative explained variance reaches the threshold.


## Step 14. Reduce and Reconstruct the Data

The worksheet transformation is:

$$
Z=X_cW_k
$$

The PCA reconstruction brings the reduced representation back to the original feature space.

**TODO:**
1. Create PCA with the selected `k`.
2. Transform the standardized data.
3. Reconstruct the data using `inverse_transform`.
4. Compute the reconstruction MSE.


In [ ]:
# TODO: Fit PCA with k components, transform the standardized data,
# reconstruct it, and compute reconstruction MSE.

print(f"Number of components kept (k): {k}")
print(f"Reconstruction MSE (on standardized features): {reconstruction_error:.5f}")


A small reconstruction error means the `k` components we kept preserve the original data very well; a large error means we discarded too much information for the threshold we chose. You can try changing `threshold` in Step 12 (e.g., to 0.90 or 0.99) and re-running to see this tradeoff directly.


## Step 15. PCA Inside a Proper Pipeline — Avoiding Leakage

The worksheet requires preprocessing to be learned from the training portion of each fold only.

Pipeline:

**StandardScaler → PCA → Logistic Regression**

**TODO:**
1. Create the binary `Placement` target.
2. Split the data into training and test sets.
3. Build the pipeline.
4. Run 5-fold cross-validation on the training set.

Do not fit the scaler or PCA on the full dataset before cross-validation.


In [ ]:
# TODO: Create y, split X_full/y into training and test sets,
# then build the leakage-safe StandardScaler -> PCA -> LogisticRegression pipeline
# and run 5-fold cross-validation on the training set.

print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])

print("Cross-validation accuracy scores (with PCA):", np.round(scores, 3))
print("Mean accuracy (with PCA):", scores.mean().round(3), "| Std:", scores.std().round(3))


## Step 16. Compare PCA vs No-PCA

Create a baseline with:

**StandardScaler → Logistic Regression**

**TODO:** Build the baseline pipeline, run the same 5-fold cross-validation, and compare its mean accuracy with the PCA pipeline.

Remember: PCA is unsupervised and maximizes variance in $X$; it does **not** directly maximize prediction accuracy for $y$.


In [ ]:
# TODO: Build the no-PCA baseline:
# StandardScaler -> LogisticRegression
# Then run the same 5-fold cross-validation and compare the mean accuracy.

print("Cross-validation accuracy scores (baseline, no PCA):", np.round(baseline_scores, 3))
print("Mean accuracy (baseline, no PCA):", baseline_scores.mean().round(3), "| Std:", baseline_scores.std().round(3))

print("\n--- Summary ---")
print(f"With PCA (k={k} components): mean accuracy = {scores.mean():.3f}")
print(f"Without PCA (5 features):    mean accuracy = {baseline_scores.mean():.3f}")


**How to interpret this comparison:**

- If accuracy with PCA is **close to** the no-PCA baseline, PCA successfully compressed the features into fewer dimensions **with little or no observed loss in cross-validated predictive performance**. This is useful because we reduced dimensionality while maintaining approximately the same observed model performance.
- If accuracy **drops noticeably** with PCA, it means the discarded components (the ones beyond `k`) still contained information useful for predicting `Placement`, even though they contributed little to the *overall variance* of the features. This is an important lesson: **PCA optimizes for explaining variance in X, not for predicting y** — so it is not guaranteed to help or preserve downstream predictive performance, and should always be validated before being used in a predictive pipeline.


## Summary

By the end of this lab, you should be able to:

- Mean-center data using $X_c=X-\mathbf{1}\mu^\top$
- Compute covariance using $S=\frac{1}{n-1}X_c^\top X_c$
- Obtain principal directions from $Sv_j=\lambda_jv_j$
- Project data using $Z=X_cW_k$
- Interpret explained variance and choose $k$
- Reconstruct data from reduced components
- Use PCA correctly inside cross-validation
- Compare PCA with a no-PCA baseline


# BONUS


### PCA-Based Image Compression and Decompression

In this bonus example, we use PCA to demonstrate **dimensionality reduction for image data**.

Each image in the Digits dataset contains **64 pixel values** (8 × 8). PCA compresses the 64-dimensional representation into only **15 principal components**, reducing the number of values needed to represent the image.

The compressed representation is obtained using:

`pca.fit_transform(X)`

We then reconstruct (decompress) the image using:

`pca.inverse_transform(X_compressed)`

The reconstructed image is an approximation of the original image. Because only 15 components are retained instead of all 64 original pixel values, some information is lost during compression. The reconstructed image lets us visually see how much information is preserved.

**Process:**

**Original Image (64 pixels) → PCA Compression (15 components) → PCA Reconstruction → Decompressed Image (64 pixels)**

The important observation is that PCA can represent the image using **fewer dimensions while preserving much of its visual information**. The 15 PCA values are the compressed representation; they are not themselves 15 image pixels.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

# Original image
digits = load_digits()
X = digits.data

# Compression: 64 pixels → 10 components
pca = PCA(n_components=15)
X_compressed = pca.fit_transform(X)

# Decompression / reconstruction: 10 components → 64 pixels
X_decompressed = pca.inverse_transform(X_compressed)

# Display
plt.figure(figsize=(6, 2))

plt.subplot(1, 3, 1)
plt.imshow(X[0].reshape(8, 8), cmap="gray")
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(X_compressed[0].reshape(3, 5), cmap="gray")
plt.title("Compressed")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(X_decompressed[0].reshape(8, 8), cmap="gray")
plt.title("Decompressed")
plt.axis("off")

plt.show()